# 多目的ベイズ最適化：Pareto解・Hypervolume・獲得関数の実践ガイド

2つの目的を同時に最大化するMOBOを、**モデル学習 → Pareto集合 → 参照点 → qLogEHVIによる2点同時提案 → 観測 → Pareto更新 → GP再学習**まで実行します。

獲得関数はBoTorch公開APIを直接使用し、Notebook内に新しい獲得関数クラスは定義しません。

関連理論: [多目的獲得関数](../../docs/theory/acquisition/07_multiobjective.md) / [MultiTask・MultiOutput](../../docs/theory/07_multitask_multioutput.md)


## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.multi_objective.logei import (
    qLogExpectedHypervolumeImprovement,
    qLogNoisyExpectedHypervolumeImprovement,
)
from botorch.acquisition.multi_objective.parego import qLogNParEGO
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from botorch.utils.multi_objective.pareto import is_non_dominated

from robotorchan.models import ModelListGP, SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### この例で学ぶ「多目的」の意味

品質Aを上げると品質Bが下がるような問題では、単一の「最高点」が存在するとは限りません。**Pareto優越**とは、他の候補と比べて全目的で同等以上、かつ少なくとも1目的で良いことです。誰にも優越されない点が**Pareto解**です。

この例は入力が1変数 `x∈[0,1]`、出力が2変数 `[f1(x),f2(x)]` の最大化問題です。入力1次元と目的2次元を混同しないよう、後で両方の図を描きます。

## 2. Two conflicting objectives

両目的をmaximize conventionへ揃えます。1つ目は`x≈0.25`、2つ目は`x≈0.75`付近を好むため、単一のbest pointではなくtrade-offを持つPareto setが生じます。

In [ ]:
def objectives(X: torch.Tensor) -> torch.Tensor:
    first = 1.0 - (X - 0.25).square()
    second = 1.0 - (X - 0.75).square()
    return torch.cat([first, second], dim=-1)


train_X = torch.linspace(0.0, 1.0, 9).unsqueeze(-1)
train_Y = objectives(train_X)
bounds = torch.tensor([[0.0], [1.0]])
print(train_X.shape, train_Y.shape)

### 2目的のトレードオフを入力空間で見る

一方の目的は `x=0.25`、もう一方は `x=0.75` で最大になります。片方の最大値だけを目指すと、もう片方の性能を犠牲にします。MOBOは重みを固定して単一スコアへまとめる以外に、Pareto集合の改善を直接目指すこともできます。

In [ ]:
plot_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
plot_Y = objectives(plot_X)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(plot_X.squeeze(-1), plot_Y[:, 0], label="Objective 1")
ax.plot(plot_X.squeeze(-1), plot_Y[:, 1], label="Objective 2")
ax.scatter(train_X.squeeze(-1), train_Y[:, 0], marker="o", s=25, label="Observed objective 1")
ax.scatter(train_X.squeeze(-1), train_Y[:, 1], marker="x", s=35, label="Observed objective 2")
ax.set(xlabel="Input x", ylabel="Objective value", title="Conflicting objectives")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 3. Multi-output surrogate is not yet MOBO

`ModelListGP`は2出力posteriorを提供しますが、それだけではPareto optimizationにはなりません。どのoutputsをobjectivesとして扱うか、reference pointをどこに置くか、どのMO utilityを使うかはdecision layerの責務です。

In [ ]:
model = ModelListGP(
    SingleTaskGP(train_X, train_Y[:, [0]]),
    SingleTaskGP(train_X, train_Y[:, [1]]),
)
model.fit()
model.eval()

probe_X = torch.tensor([[0.3], [0.7]])
with torch.no_grad():
    posterior = model.posterior(probe_X)
print("posterior mean shape:", posterior.mean.shape)

## 4. Observed Pareto front

In [ ]:
pareto_mask = is_non_dominated(train_Y)
pareto_Y = train_Y[pareto_mask]
pareto_X = train_X[pareto_mask]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_Y[:, 0], train_Y[:, 1], label="observations")
ax.scatter(pareto_Y[:, 0], pareto_Y[:, 1], marker="x", s=80, label="Pareto")
ax.set_xlabel("objective 1")
ax.set_ylabel("objective 2")
ax.legend()
plt.show()
print("Pareto X:", pareto_X.squeeze(-1))

## 5. Reference point is part of the problem definition

このsynthetic exampleでは観測objectiveの各成分最小値より0.1悪い点を使います。これは一般則ではありません。実問題ではobjective transform後の尺度や「hypervolumeとして数えたい最低水準」から設計します。

### Pareto図の読み方

横軸は目的1、縦軸は目的2です。右上ほど両目的で良い結果を表しますが、実際には片方を改善すると他方が下がる領域があります。

`is_non_dominated(train_Y)` は**観測済みの目的値**から非優越点を抽出します。GPの予測平均を使ったPareto集合ではありません。

In [ ]:
ref_point = train_Y.min(dim=0).values - 0.1
partitioning = FastNondominatedPartitioning(ref_point=ref_point, Y=train_Y)
print("reference point:", ref_point)

## 6. qLogEHVI and joint q=2 candidate generation

q=2はpointwise EHVI上位2点ではありません。2候補を同時に追加したときの**joint hypervolume improvement**をposterior samplesで評価します。

### 参照点とHypervolumeの意味

Hypervolume（HV）は、Pareto集合が**参照点より良い領域をどれだけ覆うか**を測る指標です。参照点は評価上の最低水準を表し、**目的値が最大化方向へ統一されていること**が前提です。

この例では観測値から簡易的に参照点を設定しますが、実際には品質仕様や意思決定上の基準から決めます。参照点を変えるとHVも変わるため、異なる参照点のHV値をそのまま比較しません。

In [ ]:
ehvi = qLogExpectedHypervolumeImprovement(
    model=model,
    ref_point=ref_point.tolist(),
    partitioning=partitioning,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=123),
)

candidates, ehvi_value = optimize_acqf(
    acq_function=ehvi,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert candidates.shape == torch.Size([2, 1])
print("joint candidates:", candidates.squeeze(-1))
print("qLogEHVI value:", ehvi_value)

In [ ]:
from botorch.utils.multi_objective.hypervolume import Hypervolume

hv = Hypervolume(ref_point=ref_point)
initial_hv = hv.compute(pareto_Y)
print("初期の観測Pareto集合のHypervolume:", initial_hv)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_Y[:, 0], train_Y[:, 1], label="Observed")
ax.scatter(pareto_Y[:, 0], pareto_Y[:, 1], marker="x", s=70, label="Pareto")
ax.scatter(ref_point[0], ref_point[1], marker="s", label="Reference point")
ax.set(xlabel="Objective 1", ylabel="Objective 2", title="Pareto set and reference point")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 7. Evaluate the batch and update the Pareto front

In [ ]:
candidate_Y = objectives(candidates)
updated_X = torch.cat([train_X, candidates], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)
updated_mask = is_non_dominated(updated_Y)
updated_pareto_Y = updated_Y[updated_mask]

print("candidate objectives:\n", candidate_Y)
print("Pareto points before / after:", pareto_Y.shape[0], updated_pareto_Y.shape[0])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_Y[:, 0], train_Y[:, 1], label="initial observations")
ax.scatter(candidate_Y[:, 0], candidate_Y[:, 1], marker="x", s=90, label="new batch")
ax.scatter(
    updated_pareto_Y[:, 0],
    updated_pareto_Y[:, 1],
    facecolors="none",
    edgecolors="black",
    s=100,
    label="updated Pareto",
)
ax.set_xlabel("objective 1")
ax.set_ylabel("objective 2")
ax.legend()
plt.show()

### qLogEHVIが2点をまとめて提案する理由

`q=2` は2回の単一点最適化ではありません。2つの候補を**同時に評価した場合のHypervolume改善量**をMCサンプルで評価します。近い候補を2点選ぶより、Pareto集合の異なる部分を改善できる候補が有利になる場合があります。

`qLogEHVI` は改善量の対数表現を使うため、返された獲得関数値をHypervolumeそのものと解釈しません。

## 8. qLogNEHVI: uncertainty in the baseline front

noiseがあるとobserved Pareto frontとlatent Pareto frontは一致しません。qLogNEHVIは`X_baseline`のlatent objective uncertaintyも積分します。ここではAPIとcandidate pathを同じmodelで実行確認します。

In [ ]:
nehvi = qLogNoisyExpectedHypervolumeImprovement(
    model=model,
    ref_point=ref_point.tolist(),
    X_baseline=train_X,
    sampler=SobolQMCNormalSampler(torch.Size([32]), seed=456),
    prune_baseline=False,
)
nehvi_candidate, nehvi_value = optimize_acqf(
    acq_function=nehvi,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
print("qLogNEHVI candidate:", nehvi_candidate)
print("qLogNEHVI value:", nehvi_value)

### 観測前後のHypervolumeを比較する

新しい2点を評価した後のHypervolumeを、同じ参照点で計算します。これは**観測された改善量**であり、候補提案時に計算したqLogEHVIの獲得関数値とは異なります。

In [ ]:
updated_hv = hv.compute(updated_pareto_Y)
print("更新前HV:", initial_hv)
print("更新後HV:", updated_hv)
print("観測されたHV改善:", updated_hv - initial_hv)
assert updated_hv >= initial_hv - 1e-8

## 9. qLogNParEGO: scalarization-based Pareto exploration

NParEGOはhypervolumeを直接最適化せず、multi-output posteriorをscalarized subproblemへ写します。iterationごとにweightsを変えることでfrontの異なる領域を探索します。

### 新しい観測を使ってGPを再学習する

候補を実際に測定したら、その2点を訓練データに追加してGPを作り直します。**Pareto集合の更新だけではGPの事後分布は変わりません**。次の候補提案には、更新済みモデルと更新済みPareto集合から作り直した獲得関数を使います。

In [ ]:
updated_model = ModelListGP(
    SingleTaskGP(updated_X, updated_Y[:, [0]]),
    SingleTaskGP(updated_X, updated_Y[:, [1]]),
)
updated_model.fit()
updated_model.eval()
updated_partitioning = FastNondominatedPartitioning(
    ref_point=ref_point, Y=updated_Y
)
next_ehvi = qLogExpectedHypervolumeImprovement(
    model=updated_model,
    ref_point=ref_point.tolist(),
    partitioning=updated_partitioning,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=124),
)
next_candidates, next_ehvi_value = optimize_acqf(
    acq_function=next_ehvi, bounds=bounds, q=2,
    num_restarts=4, raw_samples=64,
)
print("更新後のq=2提案:", next_candidates.squeeze(-1))
print("更新後のqLogEHVI:", next_ehvi_value)
assert updated_X.shape[0] == train_X.shape[0] + 2
assert next_candidates.shape == (2, 1)

In [ ]:
weights = torch.tensor([0.35, 0.65])
nparego = qLogNParEGO(
    model=model,
    X_baseline=train_X,
    scalarization_weights=weights,
    sampler=SobolQMCNormalSampler(torch.Size([32]), seed=789),
)
nparego_candidate, nparego_value = optimize_acqf(
    acq_function=nparego,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
print("weights:", weights)
print("qLogNParEGO candidate:", nparego_candidate)
print("qLogNParEGO value:", nparego_value)

### 再学習前後の2目的予測を比較する

モデル更新により各目的の予測平均が変化します。2目的のトレードオフ構造は維持されても、候補選択の根拠となる事後分布が更新される点がBOループの重要な部分です。

In [ ]:
with torch.no_grad():
    old_mean = model.posterior(plot_X).mean
    new_mean = updated_model.posterior(plot_X).mean
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for obj_idx, ax in enumerate(axes):
    ax.plot(plot_X.squeeze(-1), plot_Y[:, obj_idx], label="True objective")
    ax.plot(plot_X.squeeze(-1), old_mean[:, obj_idx], label="Before fit")
    ax.plot(plot_X.squeeze(-1), new_mean[:, obj_idx], label="After fit")
    ax.scatter(candidates.squeeze(-1), candidate_Y[:, obj_idx], marker="x", label="New observations")
    ax.set(xlabel="x", title=f"Objective {obj_idx + 1}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 10. The three decision semantics

```text
qLogEHVI
    current Pareto frontへのjoint hypervolume improvement

qLogNEHVI
    baseline objective / latent Pareto frontのuncertaintyも考慮

qLogNParEGO
    scalarization weightsで1つのsubproblemを作る
```

これらのacquisition valueを共通のperformance scoreとして比較しません。utility definitionが異なるためです。

## 11. Model dependence and candidate dependence are separate

このNotebookの`ModelListGP`は目的ごとに独立したGPです。一方、qLogEHVIのq=2 candidateはjoint posterior samplesで同時に価値付けされます。

- objective間dependence: surrogateが複数outputs間の相関をどうmodel化するか
- candidate間dependence: q候補のjoint posteriorをacquisitionがどう扱うか

この2つを混同しません。相関したobjectivesを明示的にmodel化したい場合はMultiTask/Kronecker系も候補です。

## 12. Important boundaries

- reference pointはtraining Yの最小値から常に自動決定できるものではありません。
- minimization objectiveは符号反転等でmaximize conventionへ揃えます。
- multi-output predictionとmulti-objective decisionは別概念です。
- black-box outcome constraintとknown candidate/input-space constraintは別レイヤーです。
- `X_pending`を含むasynchronous MOBOはBatch/Async Phaseで扱います。
- HVKGはTheory上存在しますが、現在robotorchanでintegration-testedなMO pathとは扱いません。
- many-objectiveではhypervolume partitioning costとreference-point sensitivityが増えます。

## 13. Related notebooks

- `05_model_list_gp.ipynb`: independent multi-output surrogate
- `04_multitask_gp.ipynb`: MultiTask / Kronecker posterior semantics
- `32_standard_acquisition.ipynb`: single-objective acquisition comparison

このNotebookはmulti-objective **decision workflow**を担当し、model-family説明を重複させません。

## 14. 公開APIと多目的最適化の流れ

| 段階 | 公開API | 意味 |
|---|---|---|
| 2出力モデル | `ModelListGP(SingleTaskGP(...), ...)` | 各目的のGPを組み合わせる |
| 学習 | `model.fit()` | 観測データから事後分布を推定 |
| 非優越点抽出 | `is_non_dominated()` | 観測Pareto集合を求める |
| HV評価 | `Hypervolume` | 同じ参照点で改善量を比較 |
| EHVI用分割 | `FastNondominatedPartitioning` | 非優越領域の分割 |
| q=2候補提案 | `qLogExpectedHypervolumeImprovement` + `optimize_acqf` | 同時2候補を最適化 |
| ノイズを考慮 | `qLogNoisyExpectedHypervolumeImprovement` | ベースラインの不確実性を考慮 |
| スカラー化 | `qLogNParEGO` | 重み付きの単目的部分問題 |
| 更新 | 観測→Pareto更新→`fit()`→獲得関数再生成 | 次の実験に進む |

**注意：** 目的値を測定する前に真のPareto解は分かりません。獲得関数はモデルの予測に基づき候補を選び、実験結果を得て初めて観測Pareto集合を更新します。